# 압수수색 영상 250만 번, 소아응급 전담 전문의는 하루 86명? (재현 노트북)

DataPopcorn 테크블로그 글 [datapopcorn.ai/blog/pediatric-emergency-ajou-data](https://datapopcorn.ai/blog/pediatric-emergency-ajou-data)의 재현 노트북입니다. `런타임 → 모두 실행`을 누르면 글에 나온 숫자와 차트 5개가 다시 만들어집니다.

- 데이터: 2026-10-25에 정리한 스냅샷. 공공 보고서의 표는 CSV로 옮겼고(출처 열 포함), 반응 데이터는 네이버 데이터랩 응답 원본과 유튜브 검색 결과 요약입니다.
- 이 노트북은 사건의 과실·책임을 판단하지 않습니다. 사건은 경찰 수사 중입니다.
- API 키 없이 실행됩니다. 보고서 원문(PDF)과 기사는 싣지 않고 출처 주소만 남겼습니다(README 참고).

In [ ]:
# 0. 준비: 한글 폰트 + 데이터 내려받기 (로컬에서는 DATA_DIR 환경변수로 폴더 지정)
import os, sys, urllib.request, urllib.parse
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    os.system('apt-get -qq install -y fonts-nanum > /dev/null')
for f in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf', '/System/Library/Fonts/AppleSDGothicNeo.ttc', 'C:/Windows/Fonts/malgun.ttf']:
    if os.path.exists(f):
        os.environ['KFONT'] = f; break
FILES = ['centers_timeline.csv', 'centers_specialists.csv', 'region_adequacy.csv', 'residents_pediatrics.csv', 'er_limit_messages.csv', 'youtube_top.json', 'datalab_daily_onerequest.json']
BASE = os.environ.get('DATA_DIR', 'https://raw.githubusercontent.com/team-datapopcorn/datapopcorn-assets/main/pediatric-emergency-ajou-data/data')
if BASE.startswith('http'):
    os.makedirs('data', exist_ok=True)
    for fn in FILES:
        urllib.request.urlretrieve(f'{BASE}/{urllib.parse.quote(fn)}', f'data/{fn}')
    DATA = 'data'
else:
    DATA = BASE
os.environ['DATA'] = DATA; os.environ['FIG'] = 'fig'; os.environ['OUT'] = 'summary.json'
print('데이터 폴더:', DATA, sorted(os.listdir(DATA)))

In [ ]:
import json,os,pandas as pd,numpy as np,matplotlib
matplotlib.use('Agg');import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
FONT=os.environ.get('KFONT');fm.fontManager.addfont(FONT)
plt.rcParams.update({'font.family':fm.FontProperties(fname=FONT).get_name(),'axes.unicode_minus':False,'figure.dpi':150,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
C={'main':'#E8590C','sub':'#1C7ED6','gray':'#868E96','light':'#CED4DA'}
D=os.environ.get('DATA','data');FIG=os.environ.get('FIG','fig');os.makedirs(FIG,exist_ok=True);S={}

## ① 반응: 검색 일별과 유튜브 상위 영상 (Step 2)
데이터랩 네 묶음을 한 요청으로 받은 일간 지수입니다(값이 없는 날은 0). 유튜브는 2026-10-25 검색 결과 상위 영상의 조회수이며, 영상 제목은 싣지 않았습니다.

In [ ]:
# ---------- 1. 반응: 유튜브 + 검색 일별 ----------
yt=pd.DataFrame(json.load(open(f'{D}/youtube_top.json')));yt=yt[yt.views>0]
news=yt[yt.channel.str.contains('SBS|채널A|OBS|YTN|JTBC|KBS|MBC|MBN|TV조선',regex=True)]
S['yt_top_n']=len(yt);S['yt_top_sum']=int(yt.views.sum());S['yt_news_sum']=int(news.views.sum());S['yt_max']=yt.sort_values('views',ascending=False).head(5)[['channel','views']].to_dict('records')
j=json.load(open(f'{D}/datalab_daily_onerequest.json'));dl=pd.DataFrame({r['title']:pd.Series({pd.Timestamp(x['period']):x['ratio'] for x in r['data']}) for r in j['results']}).reindex(pd.date_range('2026-09-01','2026-10-24')).fillna(0)
S['search_peak']={c:{'date':str(dl[c].idxmax().date()),'v':round(dl[c].max(),1)} for c in dl}
S['search_0930_1001']={c:[round(dl.loc['2026-09-30',c],1),round(dl.loc['2026-10-01',c],1),round(dl.loc['2026-10-02',c],1)] for c in dl}
fig,ax=plt.subplots(1,2,figsize=(13,4.4),gridspec_kw={'width_ratios':[1.2,1]})
d2=dl.loc['2026-09-20':]
ax[0].plot(d2.index,d2['아주대병원 압수수색'],color=C['main'],lw=2.2,label="'아주대병원 압수수색' 등")
ax[0].plot(d2.index,d2['응급실 압수수색'],color=C['sub'],lw=1.6,label="'응급실 압수수색' 등")
ax[0].plot(d2.index,d2['소아응급실'],color=C['gray'],lw=1.2,label="'소아응급실' 등")
for x,t,y in [('2026-09-30','9/30 압수수색',30),('2026-10-15','10/15 센터장 국회 기자회견',80),('2026-10-17','10/17 검색 정점',104)]:
    ax[0].axvline(pd.Timestamp(x),color=C['light'],lw=.8,ls=':');ax[0].text(pd.Timestamp(x),y,' '+t,fontsize=8.5,color=C['gray'])
ax[0].set_ylabel('검색지수 (기간 최고일=100)');ax[0].legend(frameon=False,loc='upper left',fontsize=9)
import matplotlib.dates as mdates;ax[0].xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'));ax[0].xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
ax[0].set_title('네이버 일별 검색 (한 요청 안에서 비교)',loc='left',fontsize=11)
top=yt.sort_values('views',ascending=False).head(8)
ax[1].barh(range(len(top))[::-1],top.views/1e4,color=[C['main'] if '채널A' in c else C['light'] for c in top.channel])
ax[1].set_yticks(range(len(top))[::-1]);ax[1].set_yticklabels([f"{c} · {p}" for c,p in zip(top.channel,top.published)],fontsize=9)
for i,v in zip(range(len(top))[::-1],top.views):ax[1].text(v/1e4+5,i,f'{v/1e4:.0f}만',va='center',fontsize=9)
ax[1].set_xlabel('조회수(만 회)');ax[1].set_title("유튜브 '아주대병원 압수수색' 검색 상위 영상",loc='left',fontsize=11)
fig.suptitle('① 관심은 압수수색 당일이 아니라 2주 뒤, 의료진 반발이 알려진 뒤에 터졌다',x=0.01,ha='left',fontsize=12.5)
fig.text(0.01,-0.04,'자료: 네이버 데이터랩(일간), 유튜브 검색 결과(2026-10-25 조회, 개인화·시점 영향). 영상 제목은 싣지 않음',fontsize=8,color=C['gray'])
plt.tight_layout();plt.savefig(f'{FIG}/fig1_reaction.png',bbox_inches='tight');plt.close()
from IPython.display import Image, display
display(Image(f'{FIG}/fig1_reaction.png'))

## ② 소아전문응급의료센터 수와 권역 공백 (Step 3)
센터 수는 보건복지부 보도자료(2026.5.15), 적정 수는 국회예산정책처 결산 분석이 인용한 연구 추정치(이한나 외 2025)입니다. 권역 운영 수는 14곳 명단으로 셌습니다.

In [ ]:
# ---------- 2. 센터 수 & 권역 ----------
ct=pd.read_csv(f'{D}/centers_timeline.csv');rg=pd.read_csv(f'{D}/region_adequacy.csv')
S['centers']=dict(zip(ct.year.astype(str),ct.centers.astype(int)));S['adequate_total']=int(rg.adequate.sum());S['zero_regions']=rg[rg.operating_2026_05==0].region.tolist()
fig,ax=plt.subplots(1,2,figsize=(13,4.2),gridspec_kw={'width_ratios':[1,1.5]})
ax[0].bar(ct.year.astype(str),ct.centers,color=[C['light']]*4+[C['main']]);ax[0].axhline(18,color=C['sub'],ls='--',lw=1.2);ax[0].text(-0.4,18.4,'적정 18곳(연구 추정)',color=C['sub'],fontsize=9)
for i,v in enumerate(ct.centers):ax[0].text(i,v+0.3,f'{v}곳',ha='center')
ax[0].set_ylim(0,21);ax[0].set_title('소아전문응급의료센터 수',loc='left',fontsize=11)
x=np.arange(len(rg));ax[1].bar(x-0.2,rg.adequate,0.4,color=C['light'],label='적정');ax[1].bar(x+0.2,rg.operating_2026_05,0.4,color=[C['main'] if v==0 else C['sub'] for v in rg.operating_2026_05],label='운영(2026.5)')
ax[1].set_xticks(x);ax[1].set_xticklabels(rg.region,fontsize=8.5,rotation=30);ax[1].legend(frameon=False,fontsize=9)
for i,v in enumerate(rg.operating_2026_05):
    if v==0:ax[1].text(i+0.2,0.1,'0',ha='center',color=C['main'],fontsize=10,fontweight='bold')
ax[1].set_title('권역별 적정 수 vs 운영 수',loc='left',fontsize=11)
fig.suptitle('② 5곳에서 14곳으로 늘었지만 적정 18곳에 못 미치고, 강원·충북·광주전남·제주엔 한 곳도 없다',x=0.01,ha='left',fontsize=12.5)
fig.text(0.01,-0.05,'자료: 보건복지부 보도자료(2026.5.15), 국회예산정책처 2025회계연도 결산 분석(적정 수: 이한나 외 2025 연구 인용). 권역 운영 수는 14곳 명단으로 집계',fontsize=8,color=C['gray'])
plt.tight_layout();plt.savefig(f'{FIG}/fig2_centers.png',bbox_inches='tight');plt.close()
from IPython.display import Image, display
display(Image(f'{FIG}/fig2_centers.png'))

## ③ 일평균 전담 전문의와 센터별 환자 (Step 4)
국회예산정책처 결산 분석의 표(보건복지부 자료)를 옮긴 값입니다. 2025년 12월까지 지정된 12곳 기준이며, 환자 수는 2025년 1~12월 평균입니다.

In [ ]:
# ---------- 3. 전담전문의 ----------
sp=pd.read_csv(f'{D}/centers_specialists.csv')
tot={k:round(sp[k].sum(),2) for k in ['specialists_2024','specialists_2025_06','specialists_2025_12']};S['specialists_total']=tot;S['target']=91
aj=sp[sp.center=='아주대병원'].iloc[0];rank=int(sp.patients_per_day_2025.rank(ascending=False)[sp.center=='아주대병원'].iloc[0])
S['ajou']={'patients_per_day':float(aj.patients_per_day_2025),'specialists_2025_12':float(aj.specialists_2025_12),'patients_rank_of_12':rank,'patients_per_specialist':round(aj.patients_per_day_2025/aj.specialists_2025_12,1)}
S['patients_per_specialist_all']=round(sp.patients_per_day_2025.sum()/sp.specialists_2025_12.sum(),1)
fig,ax=plt.subplots(1,2,figsize=(13,4.4),gridspec_kw={'width_ratios':[1,1.6]})
labs=['2024','2025.6','2025.12'];vals=list(tot.values())
ax[0].bar(labs,vals,color=[C['light'],C['light'],C['main']]);ax[0].axhline(91,color=C['sub'],ls='--',lw=1.2);ax[0].text(-0.4,92,'예산 목표 91명',color=C['sub'],fontsize=9)
for i,v in enumerate(vals):ax[0].text(i,v+1,f'{v:.1f}명',ha='center')
ax[0].set_ylim(0,105);ax[0].set_title('12개 센터 일평균 전담전문의 합',loc='left',fontsize=11)
s2=sp.sort_values('patients_per_day_2025');ax[1].barh(s2.center,s2.patients_per_day_2025,color=[C['main'] if c=='아주대병원' else C['light'] for c in s2.center])
for i,(p,sv) in enumerate(zip(s2.patients_per_day_2025,s2.specialists_2025_12)):ax[1].text(p+0.8,i,f'환자 {p:.0f}명 · 전문의 {sv:.1f}명',va='center',fontsize=8.5)
ax[1].set_xlim(0,100);ax[1].set_xlabel('하루 평균 소아 응급환자(2025)');ax[1].set_title('센터별 하루 평균 소아 환자와 전담전문의',loc='left',fontsize=11)
fig.suptitle(f"③ 전담전문의는 하루 평균 {tot['specialists_2025_12']:.0f}명(목표 91명), 아주대 센터는 환자 수 12곳 중 {rank}번째",x=0.01,ha='left',fontsize=12.5)
fig.text(0.01,-0.04,'자료: 국회예산정책처 2025회계연도 결산 분석(보건복지부 자료, 2025.12까지 지정된 12개소), 보건복지부 보도자료(일평균 환자는 2025년 1~12월)',fontsize=8,color=C['gray'])
plt.tight_layout();plt.savefig(f'{FIG}/fig3_specialists.png',bbox_inches='tight');plt.close()
from IPython.display import Image, display
display(Image(f'{FIG}/fig3_specialists.png'))

## ④ 소아청소년과 전공의 확보율 (Step 5)
국회예산정책처 결산 분석 각주의 값입니다. 확보율은 지원율과 분모가 다릅니다.

In [ ]:
# ---------- 4. 전공의 ----------
rs=pd.read_csv(f'{D}/residents_pediatrics.csv');S['residents']=dict(zip(rs.period,rs.securing_rate_pct))
fig,ax=plt.subplots(figsize=(10,3.8));ax.bar(rs.period,rs.securing_rate_pct,color=[C['light'],C['light'],C['main'],C['light'],C['light']])
for i,v in enumerate(rs.securing_rate_pct):ax.text(i,v+0.8,f'{v}%',ha='center')
ax.set_ylim(0,40);ax.set_ylabel('전공의 확보율(%)');ax.set_title('④ 소아청소년과 전공의 확보율: 2025년 상반기 2.9%, 올해 상반기 21.7%',loc='left')
fig.text(0.01,-0.05,'자료: 국회예산정책처 2025회계연도 결산 분석 각주. 확보율은 지원율과 분모가 다름',fontsize=8,color=C['gray'])
plt.tight_layout();plt.savefig(f'{FIG}/fig4_residents.png',bbox_inches='tight');plt.close()
from IPython.display import Image, display
display(Image(f'{FIG}/fig4_residents.png'))

## ⑤ 응급실 진료제한 메시지 (Step 6)
보건복지부 응급실 수용곤란 고지 통계(국회 제출, 더메디컬 2026.10.2 보도). '인력부족'은 의원실이 메시지 문구로 분류한 값입니다.

In [ ]:
# ---------- 5. 진료제한 메시지 ----------
er=pd.read_csv(f'{D}/er_limit_messages.csv');S['er_total']=int(er.messages.sum());S['er_share']=dict(zip(er.year,er.staff_share_pct))
fig,ax=plt.subplots(figsize=(10,3.9));ax.bar(er.year,er.messages/1e4,color=C['light'],label='전체 메시지(만 건)');ax.bar(er.year,er.staff_shortage/1e4,color=C['main'],label="그중 '인력부족' 분류")
for i,(m,p) in enumerate(zip(er.messages,er.staff_share_pct)):ax.text(i,m/1e4+0.3,f'{p}%',ha='center',color=C['main'])
ax.set_ylabel('만 건');ax.legend(frameon=False)
ax.set_title(f"⑤ 응급실 진료제한 메시지 {S['er_total']:,}건, 인력부족 비중은 19.2% → 올해 68.4%",loc='left')
fig.text(0.01,-0.05,"자료: 보건복지부 '응급실 수용곤란 고지 통계'(국회 제출, 더메디컬 2026.10.2 보도). 의원실이 메시지 문구로 분류, 실제 수용불가 건수와 다를 수 있음",fontsize=8,color=C['gray'])
plt.tight_layout();plt.savefig(f'{FIG}/fig5_er_messages.png',bbox_inches='tight');plt.close()

from IPython.display import Image, display
display(Image(f'{FIG}/fig5_er_messages.png'))

In [ ]:
import json
print(json.dumps(S, ensure_ascii=False, indent=1))